# BDC Training — Google Colab T4
**Self-contained notebook** — jalankan cell dari atas ke bawah secara berurutan.

| Step | Keterangan |
|------|------------|
| 1 | Mount Google Drive |
| 2 | Clone repo & ekstrak data |
| 3 | Install library |
| 4 | Setup & Import |
| 5 | Konfigurasi |
| 6 | Transforms & Augmentasi |
| 7 | Dataset & DataLoader |
| 8 | Model ConvNeXt Tiny |
| 9 | Training Loop (dengan resume) |
| 10 | Plot History |
| 11 | Evaluasi Validation |
| 12 | Test Prediction & Submission |

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive berhasil di-mount!')

## Step 2 — Clone Repo & Ekstrak Dataset

In [ ]:
import os, zipfile

%cd /content
!rm -rf /content/proyek_bdc
!git clone https://github.com/YanDraa/BDC.git /content/proyek_bdc
%cd /content/proyek_bdc

# Ekstrak dataset dari Google Drive ke SSD lokal Colab (lebih cepat saat training)
ZIP_PATH     = '/content/drive/MyDrive/BDC/data.zip'
EXTRACT_PATH = '/content/proyek_bdc'

print('Mengekstrak dataset... Mohon tunggu.')
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print('[OK] Ekstrak selesai!')

# Verifikasi struktur folder
print('\nStruktur folder data:')
for item in sorted(os.listdir(EXTRACT_PATH)):
    print(f'  {item}')
print(f"\nPath train ada? -> {os.path.exists('/content/proyek_bdc/data/train')}")
print(f"Path test  ada? -> {os.path.exists('/content/proyek_bdc/data/test')}")

## Step 3 — Install Library

In [ ]:
!pip install -q timm
print('[OK] Library siap!')

## Step 4 — Setup & Import

In [ ]:
import os, warnings, random, shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import timm
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 5 — Konfigurasi

In [ ]:
# --- Path Data (SUDAH DIPERBAIKI) ---
TRAIN_DIR = '/content/proyek_bdc/data/train'
TEST_DIR  = '/content/proyek_bdc/data/test'

# --- Google Drive (anti-disconnect) ---
DRIVE_SAVE_DIR  = '/content/drive/MyDrive/BDC/models'
EXPERIMENT_NAME = 'convnext_tiny_384'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
os.makedirs('models',       exist_ok=True)
os.makedirs('submission',   exist_ok=True)

LAST_CKPT_PATH = os.path.join(DRIVE_SAVE_DIR, f'last_{EXPERIMENT_NAME}.pth')
BEST_CKPT_PATH = os.path.join(DRIVE_SAVE_DIR, f'best_{EXPERIMENT_NAME}.pth')
HISTORY_CSV    = os.path.join(DRIVE_SAVE_DIR, f'history_{EXPERIMENT_NAME}.csv')

# --- Hyperparameter ---
IMG_SIZE    = 384
BATCH_SIZE  = 24
NUM_EPOCHS  = 15
LR          = 3e-4
NUM_CLASSES = 3
VAL_SPLIT   = 0.2

print(f'Config: IMG={IMG_SIZE} | BATCH={BATCH_SIZE} | EPOCH={NUM_EPOCHS} | LR={LR}')
print(f'Train : {TRAIN_DIR}')
print(f'Test  : {TEST_DIR}')
print(f'Drive : {DRIVE_SAVE_DIR}')

## Step 6 — Transforms & Augmentasi

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(40),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('[OK] Transforms siap!')

## Step 7 — Load Dataset & DataLoader

In [ ]:
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)

total      = len(full_dataset)
val_size   = int(VAL_SPLIT * total)
train_size = total - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Terapkan val_transform pada subset validasi
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print('[OK] Dataset siap!')
print(f'   Total   : {total:,} gambar')
print(f'   Train   : {train_size:,} | Val: {val_size:,}')
print(f'   Classes : {full_dataset.class_to_idx}')

## Step 8 — Model ConvNeXt Tiny

In [ ]:
model = timm.create_model(
    'convnext_tiny',
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_path_rate=0.2,
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[OK] Model ConvNeXt Tiny | Params: {n_params:,}')

## Step 9 — Training Loop

> **Anti-disconnect**: Setiap epoch disimpan ke Google Drive. Kalau runtime Colab putus, jalankan lagi cell ini — training akan **lanjut dari epoch terakhir** secara otomatis.

In [ ]:
best_f1     = 0.0
start_epoch = 0
history     = []

# --- Resume otomatis ---
if os.path.exists(LAST_CKPT_PATH):
    print(f'[Resume] Checkpoint ditemukan: {LAST_CKPT_PATH}')
    ckpt = torch.load(LAST_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_f1     = ckpt['best_f1']
    print(f'   Lanjut dari epoch {start_epoch + 1}/{NUM_EPOCHS} | best_f1: {best_f1:.4f}')
else:
    print('[Start] Training dari awal.')

if os.path.exists(HISTORY_CSV):
    history = pd.read_csv(HISTORY_CSV).to_dict('records')

print('\n' + '='*60)
print(f'  Training | {NUM_EPOCHS} epoch | {device}')
print('='*60 + '\n')

for epoch in range(start_epoch, NUM_EPOCHS):

    # -- Train --
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # -- Validation --
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]', leave=False):
            images  = images.to(device)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_f1   = f1_score(all_labels, all_preds, average='macro')
    avg_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {avg_loss:.4f} | Val Macro F1: {val_f1:.4f}')

    is_best = val_f1 > best_f1
    if is_best:
        best_f1 = val_f1
        print(f'  [BEST] F1 baru = {best_f1:.4f}')

    # -- Simpan checkpoint ke Drive setiap epoch --
    checkpoint = {
        'epoch'               : epoch,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_f1'             : best_f1,
    }
    torch.save(checkpoint, LAST_CKPT_PATH)

    if is_best:
        torch.save(model.state_dict(), BEST_CKPT_PATH)
        torch.save(model.state_dict(), f'models/best_{EXPERIMENT_NAME}.pth')

    # -- Log history --
    history.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_macro_f1': val_f1})
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    scheduler.step()

print('\n' + '='*60)
print(f'  Training selesai! Best Macro F1: {best_f1:.4f}')
print(f'  Model: {BEST_CKPT_PATH}')
print('='*60)

## Step 10 — Plot Training History

In [ ]:
df_h = pd.read_csv(HISTORY_CSV)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training History — {EXPERIMENT_NAME}', fontsize=13, fontweight='bold')

axes[0].plot(df_h['epoch'], df_h['train_loss'], marker='o', color='steelblue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_h['epoch'], df_h['val_macro_f1'], marker='o', color='darkorange')
axes[1].axhline(y=df_h['val_macro_f1'].max(), color='red', linestyle='--', alpha=0.7,
                label=f"Best: {df_h['val_macro_f1'].max():.4f}")
axes[1].set_title('Val Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(DRIVE_SAVE_DIR, f'history_{EXPERIMENT_NAME}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Plot disimpan: {plot_path}')

## Step 11 — Evaluasi Validation Set

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device, weights_only=True))
model.eval()

val_dataset.dataset.transform = val_transform

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating'):
        images  = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

class_names = list(full_dataset.class_to_idx.keys())
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
print(f'[OK] Final Val Macro F1: {f1_score(all_labels, all_preds, average="macro"):.4f}')

## Step 12 — Test Prediction & Submission

In [ ]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir    = root_dir
        self.transform   = transform
        self.image_files = sorted([
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]


test_dataset = TestDataset(TEST_DIR, transform=val_transform)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=False
)
print(f'[OK] Test dataset: {len(test_dataset):,} gambar')

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device, weights_only=True))
model.eval()

submission = []
with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc='Predicting'):
        images  = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        for fname, p in zip(filenames, preds):
            submission.append({'filename': fname, 'predicted': int(p)})

sub_df = pd.DataFrame(submission)
sub_df.insert(0, 'id', range(len(sub_df)))

SUBMISSION_PATH = 'submission/submission_NamaTim.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
shutil.copy(SUBMISSION_PATH, os.path.join(DRIVE_SAVE_DIR, 'submission_NamaTim.csv'))

print(f'\n[OK] Submission selesai! Total: {len(sub_df):,} prediksi')
print(f'   Lokal : {SUBMISSION_PATH}')
print(f'   Drive : {DRIVE_SAVE_DIR}/submission_NamaTim.csv')
print('\nPreview:')
print(sub_df.head(10))
print('\nDistribusi kelas:')
print(sub_df['predicted'].value_counts().sort_index())